# 💻 Notebook do Aluno — Aula 06: Pipeline RAG completo

**Disciplina:** Prompt Engineering and Artificial Intelligence  
**Instituição:** FIAP — Ciência da Computação · 2026  
**Professor:** Jorge Luiz Gomes  
**Aula 06/14 — Módulo 2: RAG · load → split → embed → retrieve → generate**  
**⏱️ 1h40min**  
**📄 PyMuPDF · RecursiveCharacterTextSplitter**  
**🔁 Andaime 50%**  

---

## 🎯 Objetivo da aula

Construir um pipeline RAG completo funcional em ~50 linhas. O LLM responde sobre documentos reais que nunca estiveram no treinamento — com citação de página e trecho. Essa é a fundação do CKP02.

---

## Como usar este notebook

- Rode as células **na ordem**, de cima para baixo (`Shift+Enter`).
- Complete apenas as partes marcadas com `___` e `👉 LACUNA`.
- Não apague o código já pronto — ele é o andaime do lab.
- Salve sua cópia: **Arquivo > Salvar uma cópia no Drive**.

## 📋 Roteiro do Lab

**Lab — Aula 06 · 2º Semestre**  
### Pipeline RAG completo sobre os PDFs do grupo ★★

*Grupo 3–4 · 20 minutos · Google Colab · PDFs do domínio obrigatórios*

1. Complete as 4 lacunas — caminhos dos PDFs, parâmetros do splitter, criação do vector store, montagem e invocação da chain.
2. Teste com 3 perguntas reais do domínio — perguntas que teriam respostas concretas nos documentos.
3. Compare sem vs. com RAG: faça a mesma pergunta ao ChatOllama direto e à chain_rag. Documente a diferença.
4. Verifique o grounding: faça uma pergunta que NÃO está nos documentos. A chain deve responder "Não encontrei essa informação" — não alucinar.

> **🎯 Gabarito das lacunas**
>
> Lacuna 1:  os caminhos dos arquivos que acabaram de ser enviados pelo upload, usados para carregar cada PDF no loader.
>
> Lacuna 2:  o tamanho de cada chunk e a sobreposição entre eles — os mesmos valores usados na aula (800 e 100) — aplicados sobre a lista de páginas carregadas.
>
> Lacuna 3:  o nome do modelo de embedding do semestre, e os chunks e embeddings passados ao vector store na criação da coleção.
>
> Lacuna 4:  a pergunta repassada sem transformação para o prompt, e a pergunta real do domínio ao invocar a chain.

---

## 🧩 Notebook Aluno — 50% de lacunas

Complete as lacunas marcadas com `___`.

In [ ]:
!pip install langchain langchain-community langchain-ollama pymupdf chromadb langchain-text-splitters -q

from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
import os
from google.colab import userdata, files

os.environ["OLLAMA_HOST"]    = "https://ollama.com"
os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY")

In [ ]:
# 👉 LACUNA 1: faça upload e carregue os PDFs do domínio
uploaded  = files.upload()
pdf_paths = [___]  # lista de caminhos dos PDFs enviados
paginas   = []
for p in pdf_paths:
    paginas.extend(PyMuPDFLoader(___).load())

# 👉 LACUNA 2: crie o splitter e divida as páginas em chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=___,     # 800 é um bom ponto de partida
    chunk_overlap=___,  # 100 para não perder contexto nas bordas
)
chunks = splitter.split_documents(___)
print(f"Chunks: {len(chunks)}")

# 👉 LACUNA 3: crie o vector store com nomic-embed-text
embeddings = OllamaEmbeddings(model=___)
db = Chroma.from_documents(___, embedding=___, persist_directory="/content/ckp02")
retriever = db.as_retriever(search_kwargs={"k":3})

# 👉 LACUNA 4: monte e invoque a chain RAG com grounding e citação
chain_rag = (
    {"contexto": retriever | RunnableLambda(formatar_contexto),
     "pergunta": ___,
     "nome_doc": RunnableLambda(lambda _: "PDFs do domínio")}
    | prompt | ChatOllama(model="gpt-oss:120b", temperature=0) | StrOutputParser()
)
print(chain_rag.invoke(___))  # sua pergunta sobre o domínio

---

## ✍️ Suas anotações

Registre aqui as observações pedidas no roteiro (qualidade dos resultados, comparações e conclusões do grupo).

## 📚 Referências da aula

- Paper Lewis, P. et al. — "Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks." NeurIPS, 2020. O paper original que cunhou o termo RAG. arxiv.org/abs/2005.11401
- Docs LangChain — RAG tutorial completo com PyMuPDF, Chroma e LCEL. python.langchain.com/docs/tutorials/rag
- Docs PyMuPDF — Documentação do loader LangChain com PyMuPDF. python.langchain.com/docs/integrations/document_loaders/pymupdf
- Docs RecursiveCharacterTextSplitter — Estratégias de chunking, parâmetros e separadores. python.langchain.com/docs/how_to/recursive_text_splitter
- Livro Goodfellow, I.; Bengio, Y.; Courville, A. — Deep Learning. Pearson, 2017. Cap. 15 — Representações distribuídas: a base teórica dos embeddings usados no RAG.

---

**→ Próxima Aula — Aula 07 · 21/09** — RAG avançado — chunking estratégico, reranking e RAGAS
  
Medir faithfulness e answer relevancy. Otimizar o pipeline. Entregar CKP02.

---

*Copyright © 2026 Prof. Jorge Luiz Gomes · FIAP · Todos os direitos reservados.*